In [42]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

mnist = fetch_openml('mnist_784', version=1)

X = mnist.data
y = mnist.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    train_size=50000,
    random_state=42
)

X_test, X_valid, y_test, y_valid = train_test_split(
    X_test, y_test,
    train_size=10000,
    random_state=42
)

print(len(X_train))
print(len(X_valid))
print(len(X_test))

50000
10000
10000


In [43]:
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier

rnd_clf = RandomForestClassifier(random_state=42, n_estimators=100, n_jobs=-1)
etrees_clf = ExtraTreesClassifier(random_state=42, n_estimators=100, n_jobs=-1)

rnd_clf.fit(X_train, y_train)
etrees_clf.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,False
,oob_score,False


In [44]:
from sklearn.ensemble import VotingClassifier

voting_clf_soft = VotingClassifier(
    estimators=[
        ('rf', rnd_clf),
        ('et', etrees_clf),
    ],
    voting='soft',
    n_jobs=-1
)

voting_clf_soft.fit(X_train, y_train)

,estimators,"[('rf', ...), ('et', ...)]"
,voting,'soft'
,weights,None
,n_jobs,-1
,flatten_transform,True
,verbose,False
,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1


In [47]:
from xgboost import XGBClassifier

xgb_clf = XGBClassifier(   
    objective='binary:logistic', 
    n_estimators=100,
    eval_metric="aucpr",
    early_stopping_rounds=10,
    random_state=42,
    device='cuda:0'
)

xgb_clf.fit(X_train, y_train.astype(int), eval_set=[(X_valid, y_valid.astype(int))])

[0]	validation_0-aucpr:0.89860
[1]	validation_0-aucpr:0.93853
[2]	validation_0-aucpr:0.95421
[3]	validation_0-aucpr:0.96254
[4]	validation_0-aucpr:0.96904
[5]	validation_0-aucpr:0.97294
[6]	validation_0-aucpr:0.97632
[7]	validation_0-aucpr:0.97830
[8]	validation_0-aucpr:0.98005
[9]	validation_0-aucpr:0.98183
[10]	validation_0-aucpr:0.98303
[11]	validation_0-aucpr:0.98408
[12]	validation_0-aucpr:0.98529
[13]	validation_0-aucpr:0.98632
[14]	validation_0-aucpr:0.98710
[15]	validation_0-aucpr:0.98795
[16]	validation_0-aucpr:0.98856
[17]	validation_0-aucpr:0.98901
[18]	validation_0-aucpr:0.98940
[19]	validation_0-aucpr:0.98993
[20]	validation_0-aucpr:0.99032
[21]	validation_0-aucpr:0.99069
[22]	validation_0-aucpr:0.99103
[23]	validation_0-aucpr:0.99131
[24]	validation_0-aucpr:0.99158
[25]	validation_0-aucpr:0.99186
[26]	validation_0-aucpr:0.99212
[27]	validation_0-aucpr:0.99238
[28]	validation_0-aucpr:0.99262
[29]	validation_0-aucpr:0.99284
[30]	validation_0-aucpr:0.99301
[31]	validation_0-

,objective,'multi:softprob'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,'cuda:0'
,early_stopping_rounds,10
,enable_categorical,False
,eval_metric,'aucpr'


In [49]:
import numpy as np

X_stacking_1 = rnd_clf.predict(X_valid)
X_stacking_2 = etrees_clf.predict(X_valid)
X_stacking = np.c_[X_stacking_1, X_stacking_2]

In [50]:
stack_clf = RandomForestClassifier(random_state=42, n_estimators=200, n_jobs=-1)
stack_clf.fit(X_stacking, y_valid)

,n_estimators,200
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [51]:
X_stacking_test = np.c_[rnd_clf.predict(X_test), etrees_clf.predict(X_test)]

print(rnd_clf.score(X_test, y_test))
print(etrees_clf.score(X_test, y_test))
print(voting_clf_soft.score(X_test, y_test))
print(stack_clf.score(X_stacking_test, y_test))
print(xgb_clf.score(X_test, y_test.astype(int)))

0.9677
0.9689
0.9682
0.9687
0.9759
